# 📓 Ensemble Models: Decision Trees, Random Forests, & XGBoost

This notebook is the hands-on companion to **Part 3** of our Ensemble Models series. We will build a complete machine learning pipeline to train, tune, and evaluate:
1. **Decision Trees** (Baseline model)
2. **Random Forests** (Bagging ensemble)
3. **XGBoost** (Gradient Boosting ensemble)

We will use a synthetic customer churn dataset to demonstrate data preprocessing, hyperparameter search using cross-validation, and model interpretation.

---  
## 🛠️ 1. Setup and Library Imports

First, we import the standard data science and machine learning libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn tools
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, 
    accuracy_score, 
    roc_auc_score, 
    roc_curve, 
    confusion_matrix
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# XGBoost
from xgboost import XGBClassifier

# Formatting
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
np.random.seed(42)

---  
## 📊 2. Synthetic Dataset Generation

To simulate a real-world customer churn scenario, we will generate a dataset containing both continuous and categorical features. We will also inject some missing values (`NaN`) to show how different models handle them.

In [ ]:
# 1. Create a classification dataset with 20 features
X_raw, y_raw = make_classification(
    n_samples=2000, 
    n_features=12, 
    n_informative=8, 
    n_redundant=4, 
    random_state=42
)

# 2. Convert to a DataFrame and add descriptive column names
feature_names = [
    "age", "tenure", "monthly_charges", "total_charges",
    "support_tickets", "login_frequency", "usage_metric_a", "usage_metric_b",
    "usage_metric_c", "usage_metric_d", "usage_metric_e", "usage_metric_f"
]
df = pd.DataFrame(X_raw, columns=feature_names)
df["churn"] = y_raw

# 3. Convert some numeric columns to categories
# E.g., splitting a feature into 'contract_type' and 'payment_method'
df["contract_type"] = pd.qcut(df["usage_metric_e"], q=3, labels=["Month-to-month", "One year", "Two year"])
df["payment_method"] = pd.qcut(df["usage_metric_f"], q=3, labels=["Electronic check", "Credit card", "Bank transfer"])

# Drop the original columns used for binning
df = df.drop(columns=["usage_metric_e", "usage_metric_f"])

# 4. Inject missing values (NaN) into 'monthly_charges' and 'tenure'
# E.g., simulating incomplete records
mask_charges = np.random.rand(*df["monthly_charges"].shape) < 0.08
df.loc[mask_charges, "monthly_charges"] = np.nan

mask_tenure = np.random.rand(*df["tenure"].shape) < 0.05
df.loc[mask_tenure, "tenure"] = np.nan

df.head()

---  
## 🧪 3. Data Splitting & Pipeline Preprocessing

We separate our target column `churn` and split our dataset into **80% Training** and **20% Testing**.

Since tree models are scale-invariant, we **do not need to scale our features**. We only need to:
1. **Impute missing values** (for Scikit-Learn trees).
2. **Encode categorical values** into numbers.

### 💡 Concept Check: Pipelines, ColumnTransformers, & Tree Scale Invariance

#### 1. Why Use Pipelines and ColumnTransformers?
*   **Preventing Data Leakage:** 
    A common mistake in machine learning is preprocessing the entire dataset (e.g., computing the mean for imputation or scaling) *before* splitting the data. This leaks information from the test set into the training set, leading to overly optimistic test performance.
*   **Encapsulation:** 
    A `Pipeline` binds transformers (imputation, encoding) and estimators (classifiers) into a single object. When you call `.fit()`, it fits transformations on the training data only. When you call `.predict()`, it applies those pre-fit transformations to the test data.
*   **ColumnTransformer:** 
    Since different columns require different transformations (e.g., imputing numeric columns vs. one-hot encoding categorical columns), `ColumnTransformer` allows us to apply targeted pipelines to specific column subsets simultaneously.

#### 2. Why Decision Trees Are Scale-Invariant
Unlike linear regression, SVMs, or neural networks, **tree-based models do not require features to be scaled** (e.g., using `StandardScaler` or `MinMaxScaler`). 

*   **Monotonic Splitting:**
    Decision trees split features by evaluating thresholds of the form:
    $$X_j \le \theta$$
    This split divides the data into two groups based on the relative ordering of the values, not their absolute magnitudes.
*   **Mathematical Equivalence under Scaling:**
    If we scale a feature $X_j$ using a monotonic transformation (like standardization $X_j' = \frac{X_j - \mu}{\sigma}$), the optimal split threshold $\theta$ simply scales to $\theta' = \frac{\theta - \mu}{\sigma}$. 
    *   The subset of samples routed to the left child ($X_j' \le \theta'$) is **exactly the same** as the subset routed to the left child under unscaled data ($X_j \le \theta$).
    *   Since the sample splits are identical, the Gini impurity, Entropy, and Information Gain remain mathematically unchanged.
*   **Contrast with Other Models:**
    *   **Distance-based models (KNN, KMeans, SVM)**: Use Euclidean distance ($\sqrt{\sum (x_i - y_i)^2}$). If one feature ranges from $0$ to $100000$ (e.g., Income) and another ranges from $0$ to $1$ (e.g., Churn), the distance will be dominated entirely by the Income feature.
    *   **Gradient descent models (Logistic Regression, Neural Nets)**: Unequal scales cause the gradient updates to oscillate, making optimization slow or unstable.

*Therefore, while we scale features in our pipeline as a best practice when comparing trees to other models, it has **zero mathematical impact** on the Decision Tree, Random Forest, or XGBoost results.*


In [ ]:
# Split target and features
X = df.drop(columns=["churn"])
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")

### Creating the Preprocessing Pipeline

In [ ]:
# Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

# 1. Numeric pipeline: Impute missing values with the median
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# 2. Categorical pipeline: Ordinal Encode categorical columns (assigning integer labels)
categorical_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Preprocess the datasets
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Get updated feature order for interpreting feature importances later
all_features = numeric_features + categorical_features

---  
## 🧮 3.5 Math Checkpoint: Gini, Entropy, & Split Gain Calculations

Before training a model, let's understand the math behind split selection. A Decision Tree scans all features and thresholds to find the split that maximizes **Impurity Gain**.

### Formulas:
- **Gini Impurity**:
  $$Gini(D) = 1 - \sum_{i=1}^{C} p_i^2$$
- **Entropy**:
  $$Entropy(D) = - \sum_{i=1}^{C} p_i \log_2(p_i)$$
- **Split Gain (Information Gain)**:
  $$Gain = Impurity(Parent) - \left[ \frac{N_{left}}{N_{parent}} Impurity(Left) + \frac{N_{right}}{N_{parent}} Impurity(Right) \right]$$

In [ ]:
# 1. Define Gini and Entropy functions in Python
def calculate_gini(labels):
    labels = np.array(labels)
    if len(labels) == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)
    probabilities = counts / len(labels)
    return 1.0 - np.sum(probabilities ** 2)

def calculate_entropy(labels):
    labels = np.array(labels)
    if len(labels) == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)
    probabilities = counts / len(labels)
    return -np.sum([p * np.log2(p) for p in probabilities if p > 0])

# 2. Calculate impurity for a Parent Node containing 5 Churners (1) and 5 Active users (0)
parent_labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
parent_gini = calculate_gini(parent_labels)
parent_entropy = calculate_entropy(parent_labels)

print(f"Parent Node (5 Churn, 5 Active):")
print(f"  Gini Impurity: {parent_gini:.4f}")
print(f"  Entropy:       {parent_entropy:.4f}\n")

### Simulating a Split
Let's say we split the parent node on a feature (e.g. Tenure). The split results in:
- **Left Child**: 5 Churners, 0 Active users (a pure churn node).
- **Right Child**: 0 Churners, 5 Active users (a pure active node).

In [ ]:
left_child = [1, 1, 1, 1, 1]
right_child = [0, 0, 0, 0, 0]

left_gini = calculate_gini(left_child)
right_gini = calculate_gini(right_child)

# Weighted Impurity of Children
n_parent = len(parent_labels)
n_left = len(left_child)
n_right = len(right_child)

weighted_child_gini = (n_left / n_parent) * left_gini + (n_right / n_parent) * right_gini
gini_gain = parent_gini - weighted_child_gini

print(f"Children Node Impurities:")
print(f"  Left Child Gini: {left_gini:.4f}")
print(f"  Right Child Gini: {right_gini:.4f}")
print(f"  Gini Gain of this split: {gini_gain:.4f} (Max possible gain!)\n")

### Continuous Feature Splits (Scanning Multiple Thresholds)
How does a tree scan a continuous column? Let's say we have a feature `monthly_charges` and we want to find the best threshold split. We will evaluate Gini Gain at multiple thresholds.

In [ ]:
# Toy dataset of 10 customers
toy_df = pd.DataFrame({
    'monthly_charges': [25, 35, 45, 55, 65, 75, 85, 95, 105, 115],
    'churn':           [0,  0,  0,  1,  0,  1,  1,  1,  1,   1]
})

parent_gini = calculate_gini(toy_df['churn'])
print(f"Parent Churn Gini: {parent_gini:.4f}\n")

# Scan multiple possible split thresholds
results = []
possible_thresholds = [30, 50, 70, 90, 100]

for thresh in possible_thresholds:
    left_labels = toy_df[toy_df['monthly_charges'] <= thresh]['churn']
    right_labels = toy_df[toy_df['monthly_charges'] > thresh]['churn']
    
    left_gini = calculate_gini(left_labels)
    right_gini = calculate_gini(right_labels)
    
    # Weighted child Gini
    w_gini = (len(left_labels)/len(toy_df))*left_gini + (len(right_labels)/len(toy_df))*right_gini
    gain = parent_gini - w_gini
    
    results.append({
        'Threshold': thresh,
        'Left Gini': left_gini,
        'Right Gini': right_gini,
        'Gini Gain': gain
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

best_row = results_df.loc[results_df['Gini Gain'].idxmax()]
print(f"\nBest split threshold is monthly_charges > {best_row['Threshold']} with Gini Gain of {best_row['Gini Gain']:.4f}")

---  
## 🌳 4. Baseline: Decision Tree

We train a simple decision tree with a `max_depth` limit of 3 to visualize how the splitting rules are constructed.

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_model.fit(X_train_preprocessed, y_train)

# Plot the tree structure
plt.figure(figsize=(18, 10))
plot_tree(
    dt_model, 
    feature_names=all_features, 
    class_names=["Stay", "Churn"], 
    filled=True, 
    rounded=True, 
    fontsize=12
)
plt.title("Visualizing Decision Tree Splits (Depth=3)", fontsize=16)
plt.show()

---  
## 🌲 5. Parallel Ensembles: Random Forest & Hyperparameter Tuning

We will tune a **Random Forest Classifier** using **RandomizedSearchCV** with a 5-Fold cross-validation strategy. We will also access the built-in **Out-of-Bag (OOB) Score**.

### 💡 Scikit-Learn Class and Function Explanations

#### 1. `RandomForestClassifier`
A Random Forest is a parallel ensemble of independent decision trees. It uses **bagging** (Bootstrap Aggregation) and **feature subspace sampling** to ensure the individual trees are diverse and decorrelated.

*   **Key Parameters:**
    *   `n_estimators`: The number of decision trees to build. More trees reduce variance and increase stability but take longer to train.
    *   `max_depth`: The maximum depth of each tree. Shallow trees have high bias; deep trees can overfit. `None` splits nodes until all leaves are pure or contain fewer than `min_samples_split` samples.
    *   `max_features`: The size of the random subset of features to inspect at each split. Typical classifications use `'sqrt'` (square root of total features).
    *   `min_samples_split` / `min_samples_leaf`: The minimum samples required to split an internal node or be at a leaf node. Higher values prevent deep, noisy splits.
    *   `bootstrap`: If `True`, draws random training samples with replacement for each tree.
    *   `oob_score`: If `True`, computes the Out-of-Bag (OOB) score. This uses the leftover training samples (approx. 36.8%) not seen by a given tree to validate its performance. It acts as a built-in cross-validation.

#### 2. `RandomizedSearchCV`
Instead of exhaustively checking every combination in a grid (which is computationally slow), `RandomizedSearchCV` samples a fixed number of parameter settings (`n_iter`) from specified distributions. This is mathematically proven to find excellent hyperparameters in a fraction of the time.

*   **Key Parameters:**
    *   `estimator`: The model instance to tune (e.g., `rf_base`).
    *   `param_distributions`: Dictionary where keys are parameters and values are lists/distributions to sample from.
    *   `n_iter`: The number of random parameter combinations to try.
    *   `cv`: Number of cross-validation folds. E.g., `cv=5` splits training data into 5 folds.
    *   `scoring`: Evaluation metric (e.g., `'roc_auc'`).
    *   `n_jobs`: Number of CPU cores to run in parallel. `-1` uses all available cores.


In [ ]:
# Define the parameter grid to sample from
rf_param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [5, 10, 15, 20, None],
    'max_features': ['sqrt', 'log2', 0.5, 0.8],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize base model with bootstrap enabled to compute OOB score
rf_base = RandomForestClassifier(bootstrap=True, oob_score=True, random_state=42)

# Setup Randomized Search
rf_search = RandomizedSearchCV(
    estimator=rf_base, 
    param_distributions=rf_param_grid,
    n_iter=20, 
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train_preprocessed, y_train)

print(f"Best Random Forest Parameters: {rf_search.best_params_}")
print(f"Best Cross-Validation ROC-AUC: {rf_search.best_score_:.4f}")

### OOB (Out-of-Bag) Score Verification
Let's extract the best estimator and check its OOB score. Since OOB acts as a built-in validation set, it should be very close to the cross-validation score.

In [ ]:
best_rf = rf_search.best_estimator_
print(f"Random Forest OOB Accuracy Score: {best_rf.oob_score_:.4f}")

---  
## ⚡ 6. Sequential Ensembles: XGBoost & Native Missing Data

XGBoost handles missing values (`NaN`) natively. To demonstrate this, we will train XGBoost using **raw data** (where categorical strings are Ordinal Encoded, but numeric missing values are **not imputed**).

### 💡 Concept Check: XGBoost's Sparsity-Aware Split Finding (Native NaN Handling)

In real-world datasets, missing values (`NaN`) are extremely common. Traditional tree implementations (like Scikit-Learn's `DecisionTreeClassifier` or `RandomForestClassifier`) cannot handle missing values natively. If they encounter `NaN` values, they throw a runtime error, forcing you to manually impute them (e.g., using `SimpleImputer` with mean or median).

XGBoost solves this by introducing a **sparsity-aware split finding algorithm** that handles missing values natively, efficiently, and often with better accuracy than manual imputation.

#### How It Works Under the Hood:
When building a tree and evaluating a split threshold on feature $X_j$:
1.  **Isolate Non-Missing Values:** XGBoost temporarily ignores all samples where feature $X_j$ is missing (`NaN`).
2.  **Evaluate Left Routing:** 
    It evaluates the split gain under the assumption that **all missing values are routed to the left child node**.
3.  **Evaluate Right Routing:** 
    It evaluates the split gain under the assumption that **all missing values are routed to the right child node**.
4.  **Select the Best Path:** 
    It compares the two gains and chooses the default direction (left or right) that yields the **highest split Gain**.
5.  **Inference Routing:** 
    This default direction is saved inside the node. At prediction time, if a test sample has a missing value for $X_j$, it is automatically routed along this saved default direction.

#### Why This is Better Than Manual Imputation:
*   **Saves Computation:** You do not need to spend time or memory calculating and applying imputations.
*   **Handles Missingness as a Signal:** Sometimes, the fact that a value is missing is itself a strong predictor (e.g., a customer not filling out an optional field). XGBoost automatically captures this signal by grouping all missing values together in the optimal branch, rather than washing it out by filling them with the median or mean.


In [ ]:
# Preprocess data WITHOUT numeric imputation
xgb_preprocessor = ColumnTransformer(transformers=[
    ('num', 'passthrough', numeric_features), # Leave NaNs intact
    ('cat', categorical_transformer, categorical_features)
])

X_train_xgb = xgb_preprocessor.fit_transform(X_train)
X_test_xgb = xgb_preprocessor.transform(X_test)

### Tuning XGBoost
We define the tuning space for XGBoost. We tune `learning_rate`, `max_depth`, row/column subsampling, and regularization weights.

### 💡 XGBoost Class and Parameter Explanations

#### `XGBClassifier`
Unlike Random Forests, XGBoost is a sequential ensemble that builds trees one after the other. Each new tree is trained to predict the **residuals** (errors) of the cumulative ensemble. It uses second-order Taylor expansion gradients ($g_i$ and $h_i$) to optimize custom loss objectives under L1/L2 regularization.

*   **Key Parameters:**
    *   `n_estimators`: The number of sequential boosting rounds (trees) to build. Too many can cause overfitting unless paired with a lower learning rate.
    *   `learning_rate` (also known as `eta`): Step-size shrinkage applied to each tree's weights after boosting rounds. Lower values (e.g., `0.01` to `0.1`) make model updates more robust but require more `n_estimators`.
    *   `max_depth`: The maximum depth of each tree. Unlike Random Forests, XGBoost trees are typically shallow (e.g., 3 to 9) since they build incrementally on top of each other.
    *   `subsample`: The fraction of training rows sampled randomly before building each tree (prevents overfitting).
    *   `colsample_bytree`: The fraction of columns (features) sampled randomly before building each tree.
    *   `reg_lambda` (L2 penalty) / `reg_alpha` (L1 penalty): Regularization terms on leaf weights. L2 smooths leaf weights, while L1 encourages sparsity (driving inactive weights to exactly zero).


In [ ]:
xgb_param_grid = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_lambda': [0.1, 1.0, 10.0],
    'reg_alpha': [0.0, 0.1, 1.0],
    'n_estimators': [100, 200, 300]
}

xgb_base = XGBClassifier(
    use_label_encoder=False, 
    eval_metric='logloss', 
    random_state=42
)

xgb_search = RandomizedSearchCV(
    estimator=xgb_base, 
    param_distributions=xgb_param_grid,
    n_iter=20, 
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train_xgb, y_train)

print(f"Best XGBoost Parameters: {xgb_search.best_params_}")
print(f"Best Cross-Validation ROC-AUC: {xgb_search.best_score_:.4f}")

---  
## 📊 7. Model Comparison on Test Data

Let's compare our models on the held-out test dataset.

In [ ]:
# Get predictions
dt_preds = dt_model.predict(X_test_preprocessed)
rf_preds = best_rf.predict(X_test_preprocessed)
xgb_preds = xgb_search.best_estimator_.predict(X_test_xgb)

print("=== Decision Tree Classification Report ===")
print(classification_report(y_test, dt_preds))

print("=== Random Forest Classification Report ===")
print(classification_report(y_test, rf_preds))

print("=== XGBoost Classification Report ===")
print(classification_report(y_test, xgb_preds))

### 💡 Concept Check: `predict()` vs. `predict_proba()` & Accuracy vs. ROC-AUC

When comparing models on test data, it is critical to understand the distinction between the methods used to generate predictions and the metrics used to evaluate them.

#### 1. Prediction Methods: Hard vs. Soft Decisions
*   **`predict()` (Hard Labels):** 
    Returns the final class prediction (e.g., `0` or `1`). Under the hood, Scikit-Learn and XGBoost apply a fixed threshold (usually `0.5`) to the underlying probability score:
    $$\hat{y} = \begin{cases} 1 & \text{if } P(y=1|X) \ge 0.5 \\ 0 & \text{otherwise} \end{cases}$$
*   **`predict_proba()` (Soft Probability Scores):** 
    Returns the raw continuous probability score for each class (values between `0.0` and `1.0`). This is crucial because it indicates the model's **confidence** in its prediction.

#### 2. Evaluation Metrics: Accuracy vs. ROC-AUC
*   **Accuracy (from `predict()`):**
    Calculates the proportion of correctly predicted hard labels on the test set. While easy to interpret, accuracy is heavily dependent on a single threshold (`0.5`) and is highly misleading when classes are imbalanced.
*   **ROC-AUC (from `predict_proba()`):**
    The **Receiver Operating Characteristic** curves plot the True Positive Rate (Sensitivity) against the False Positive Rate (1 - Specificity) across **every possible threshold** from `0.0` to `1.0`. 
    The **Area Under the Curve (AUC)** measures the probability that a randomly chosen positive sample will be ranked higher than a randomly chosen negative sample. An AUC of `1.0` is perfect; `0.5` represents random guessing.

#### 3. Why XGBoost Tends to Have a Higher AUC
*   **Random Forest** is an ensemble of independent trees. The output probability is the fraction of trees voting for a class. Since the trees are built independently, leaf probabilities tend to average out towards the center, sometimes leading to less calibrated boundary probabilities.
*   **XGBoost** builds trees sequentially to directly minimize the loss function objective using first and second derivatives. This mathematical focus on gradients leads to **sharper, better-calibrated probability scores** and superior sample ranking.
*   As a result, even if Random Forest and XGBoost have comparable classification report accuracies at a threshold of `0.5`, **XGBoost typically exhibits a higher ROC-AUC** because it is significantly better at ordering the test samples from lowest to highest risk.


### Plotting ROC Curves

In [ ]:
# Get probability scores
dt_probs = dt_model.predict_proba(X_test_preprocessed)[:, 1]
rf_probs = best_rf.predict_proba(X_test_preprocessed)[:, 1]
xgb_probs = xgb_search.best_estimator_.predict_proba(X_test_xgb)[:, 1]

# Compute ROC statistics
dt_fpr, dt_tpr, _ = roc_curve(y_test, dt_probs)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_probs)

# Plot
plt.figure(figsize=(10, 7))
plt.plot(dt_fpr, dt_tpr, label=f"Decision Tree (AUC = {roc_auc_score(y_test, dt_probs):.4f})")
plt.plot(rf_fpr, rf_tpr, label=f"Random Forest (AUC = {roc_auc_score(y_test, rf_probs):.4f})")
plt.plot(xgb_fpr, xgb_tpr, label=f"XGBoost (AUC = {roc_auc_score(y_test, xgb_probs):.4f})")
plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves Comparison", fontsize=16)
plt.legend(loc="lower right")
plt.show()

---  
## 🔍 8. Model Interpretation: Feature Importances

Let's check and compare the feature importance calculations from both Random Forest and XGBoost.

In [ ]:
# Extract importances
rf_importances = best_rf.feature_importances_
xgb_importances = xgb_search.best_estimator_.feature_importances_

# Combine into a DataFrame
importance_df = pd.DataFrame({
    'Feature': all_features,
    'Random Forest': rf_importances,
    'XGBoost': xgb_importances
}).melt(id_vars='Feature', var_name='Model', value_name='Importance')

# Plot comparison
plt.figure(figsize=(12, 8))
sns.barplot(data=importance_df, x='Importance', y='Feature', hue='Model')
plt.title("Feature Importance Comparison: RF vs. XGBoost", fontsize=16)
plt.xlabel("Relative Importance")
plt.show()

---  
## 🏁 9. Key Takeaways

1. **Decision Tree (Baseline)**: Simple and highly interpretable, but lacks performance on complex datasets.
2. **Random Forest (Bagging)**: Combats variance by training independent deep trees. Very robust and easy to tune, but struggles slightly with native missing values.
3. **XGBoost (Boosting)**: Sequential model optimizing residuals. Achieved the highest performance (AUC), handles missing values natively, and runs extremely fast due to parallel split optimizations. However, it requires careful hyperparameter tuning to prevent overfitting.

## Appendix: Taylor Series & XGBoost Optimization Math Deep Dive

In **XGBoost (Extreme Gradient Boosting)**, the **second-order Taylor approximation** is used to simplify the model's complex objective function during training.

Instead of dealing with a mathematically difficult, non-linear loss function directly, XGBoost uses a Taylor series to turn any twice-differentiable loss function into a straightforward quadratic equation. This trick is what makes XGBoost incredibly fast and universally adaptable to various types of machine learning tasks.

---

#### The Objective Function Breakdown

When XGBoost trains its $t$-th decision tree ($f_t$), it tries to minimize a total objective function composed of training loss and a tree complexity penalty (regularization):

$$\text{Obj}^{(t)} = \sum_{i=1}^{n} L\big(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\big) + \Omega(f_t)$$

Where:
* $y_i$ is the actual target value.
* $\hat{y}_i^{(t-1)}$ is the prediction made by all previous trees combined (this is a fixed constant at step $t$).
* $f_t(x_i)$ is the new prediction step coming from the current tree we are trying to build.
* $\Omega(f_t)$ is the regularization penalty to prevent overfitting.

---

#### Mapping to the Taylor Series

To see how Taylor series fits here, look at the loss term $L\big(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\big)$.

Recall the generic single-variable Taylor expansion centered around $a$ with a step change $\Delta x$:
$$f(a + \Delta x) \approx f(a) + f'(a)\Delta x + \frac{1}{2}f''(a)\Delta x^2$$

XGBoost maps its variables directly into this structure:
* **The Center Point ($a$):** $\hat{y}_i^{(t-1)}$ (the cumulative prediction we already have).
* **The Step Change ($\Delta x$):** $f_t(x_i)$ (the output of our new tree).

Applying the expansion yields:
$$L\big(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\big) \approx L\big(y_i, \hat{y}_i^{(t-1)}\big) + g_i f_t(x_i) + \frac{1}{2} h_i f_t(x_i)^2$$

Where the derivatives are evaluated at the center point $\hat{y}_i^{(t-1)}$:
* $g_i$ is the **Gradient** (1st derivative of the loss function). It dictates the direction to adjust predictions.
* $h_i$ is the **Hessian** (2nd derivative of the loss function). It dictates the curvature (how quickly the slope changes).

---

#### Why This Simplification Matters

1. **Constants Drop Out:** The term $L(y_i, \hat{y}_i^{(t-1)})$ represents the error from trees already built. Because it cannot be altered by the new tree, it behaves as a constant and can be completely discarded from the optimization step.
2. **The Quadratic Simplification:** The remaining objective simplifies down to:
   $$\text{Obj}^{(t)} \approx \sum_{i=1}^{n} \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t(x_i)^2 \right] + \Omega(f_t)$$

---

#### 1. Finding the Optimal Leaf Weights

Each tree maps data points into a finite set of $T$ leaf nodes. If a data point falls into a specific leaf $j$, it is assigned a weight value $w_j$ ($f_t(x_i) = w_j$).

By grouping all individual data points by the specific leaf node they land in, and expanding the regularization term $\Omega(f_t) = \gamma T + \frac{1}{2}\lambda \sum w_j^2$, the formula updates to:

$$\text{Obj}^{(t)} \approx \sum_{j=1}^{T} \left[ \left(\sum_{i \in I_j} g_i\right) w_j + \frac{1}{2} \left(\sum_{i \in I_j} h_i + \lambda\right) w_j^2 \right] + \gamma T$$

Let $G_j = \sum_{i \in I_j} g_i$ (sum of gradients in leaf $j$) and $H_j = \sum_{i \in I_j} h_i$ (sum of Hessians in leaf $j$). The objective for leaf $j$ becomes a simple quadratic parabola:
$$\text{Obj}_j = G_j w_j + \frac{1}{2}(H_j + \lambda)w_j^2$$

To find the lowest point of this parabola, take the derivative with respect to $w_j$, set it to zero, and solve for the optimal weight $w_j^*$:

$$G_j + (H_j + \lambda)w_j = 0 \implies \mathbf{w_j^* = -\frac{G_j}{H_j + \lambda}}$$

---

#### 2. Calculating the Optimal Tree Score

Plugging the optimal leaf weight $w_j^*$ back into the quadratic equation reveals the best possible objective score for a given tree layout:

$$\text{Obj}^* = -\frac{1}{2} \sum_{j=1}^{T} \frac{G_j^2}{H_j + \lambda} + \gamma T$$

XGBoost uses this exact structured formula to evaluate how well a split divides the data, choosing splits that maximize the reduction of this score.

---

#### 3. Why XGBoost Uses 2nd-Order Approximation

* **Decouples Loss from Architecture:** Traditionally, changing a loss function meant rewriting the tree-splitting logic. Because of the Taylor expansion, XGBoost's core engine only needs to know two numbers for any data point: $g_i$ and $h_i$. You can plug in custom loss functions effortlessly.
* **Faster Convergence via Newton's Method:** Traditional gradient boosting relies entirely on first-order gradients (Gradient Descent). By retaining the second-order Hessian ($h_i$), XGBoost leverages Newton's optimization method. The Hessian serves as a dynamic, customized learning rate per leaf, yielding faster convergence with fewer trees.

---

#### Summary of XGBoost's Taylor Steps

| Variable | Mathematical Meaning | Role in XGBoost Optimization |
| :--- | :--- | :--- |
| **Center ($a$)** | Where the approximation is centered | The accumulated prediction of all existing trees ($\hat{y}_i^{(t-1)}$) |
| **Step ($\Delta x$)** | The local displacement value | The prediction output of the active tree being trained ($f_t(x_i)$) |
| **Gradient ($g_i$)** | First-order derivative of the loss function | Determines the ideal directional adjustment for predictions |
| **Hessian ($h_i$)** | Second-order derivative of the loss function | Gauges loss curvature to scale step sizes dynamically |
